In [ ]:
!pip install openml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.4 MB/s eta 0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=fa5fb8cdff9d6e256acdb880d0077db96082885bf9a122796d53a22208736b43
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [ ]:
from google.colab import userdata
from huggingface_hub import login
llama_token = userdata.get('llama_token')
login(llama_token)

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm
import openml


In [ ]:
# Настройка промпта для Credit_G
prompt_config = {
        "task": "Classify credit risk as good or bad",
        "pos_label": "good",
        "neg_label": "bad",
        "entity": "Client",
        "question": "Is this client a good credit risk?"
}

openml_id = 31

output_dir = "/content/drive/MyDrive/finetuned_llama_lora_with_Credit_G"

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464, prompt_config=None):
    dataset = openml.datasets.get_dataset(openml_id)
    target_col = dataset.default_target_attribute
    X, y, _, _ = dataset.get_data(dataset_format="dataframe", target=target_col)

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):
    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name = load_dataset(openml_id)
train_df, val_df, test_df = split_dataset(df, target_name)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   checking_status         1000 non-null   category
 1   duration                1000 non-null   uint8   
 2   credit_history          1000 non-null   category
 3   purpose                 1000 non-null   category
 4   credit_amount           1000 non-null   float64 
 5   savings_status          1000 non-null   category
 6   employment              1000 non-null   category
 7   installment_commitment  1000 non-null   uint8   
 8   personal_status         1000 non-null   category
 9   other_parties           1000 non-null   category
 10  residence_since         1000 non-null   uint8   
 11  property_magnitude      1000 non-null   category
 12  age                     1000 non-null   uint8   
 13  other_payment_plans     1000 non-null   category
 14  housing                 1

In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  {prompt_config['neg_label']}: {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  {prompt_config['pos_label']}: {counts[1]:5d} — {pcts[1]:.1f}%")


train (всего: 600):
  bad:   180 — 30.0%
  good:   420 — 70.0%

val (всего: 200):
  bad:    60 — 30.0%
  good:   140 — 70.0%

test (всего: 200):
  bad:    60 — 30.0%
  good:   140 — 70.0%


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names, target_name, prompt_config=None, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    if include_target and prompt_config is not None:
        target_value = prompt_config['pos_label'] if row[target_name] == 1 else prompt_config['neg_label']
        text += f": {target_name} -> {target_value}"

    return text

# Тест
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, True))
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, False))
train_df.head(1)

The category of checking_status is >=200. The value of duration is 12. The category of credit_history is existing paid. The category of purpose is furniture/equipment. The value of credit_amount is 2251.00. The category of savings_status is <100. The category of employment is 1<=X<4. The value of installment_commitment is 1. The category of personal_status is female div/dep/mar. The category of other_parties is none. The value of residence_since is 2. The category of property_magnitude is car. The value of age is 46. The category of other_payment_plans is none. The category of housing is own. The value of existing_credits is 1. The category of job is unskilled resident. The value of num_dependents is 1. The category of own_telephone is none. The category of foreign_worker is yes.: class -> good
The category of checking_status is >=200. The value of duration is 12. The category of credit_history is existing paid. The category of purpose is furniture/equipment. The value of credit_amount

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
251,>=200,12,existing paid,furniture/equipment,2251.0,<100,1<=X<4,1,female div/dep/mar,none,...,car,46,none,own,1,unskilled resident,1,none,yes,1


In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip()
    response = response.rstrip('.,!? ')

    pos = prompt_config['pos_label'].lower()
    neg = prompt_config['neg_label'].lower()

    # Точное совпадение
    if response == pos:
        return 1
    elif response == neg:
        return 0

    # Начинается с метки
    if response.startswith(pos):
        return 1
    elif response.startswith(neg):
        return 0

    # Содержит метку как отдельное слово
    words = response.split()
    if pos in words:
        return 1
    elif neg in words:
        return 0

    # Не удалось распознать
    print(f"Warning: Could not parse '{response}' (expected '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}')")
    return 0



response = prompt_config['pos_label']
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "hi"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: 'good'
Prediction: 1

Response: 'hi'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    pr  = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    return roc, f1, acc, pr, rec

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name, prompt_config)
print(f"\nПример текста:\n{sample_text[:300]}")


Пример текста:
The category of checking_status is 0<=X<200. The value of duration is 18. The category of credit_history is delayed previously. The category of purpose is business. The value of credit_amount is 2427.00. The category of savings_status is no known savings. The category of employment is >=7. The value


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Название модели: Llama 3.1 8B Instruct от Meta
model_name = "meta-llama/Llama-3.1-8B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

# Загрузка токенизатора для Llama 3.1
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    allocated_memory = torch.cuda.memory_allocated() / 1e9
    print(f"Занято видеопамяти на GPU: {allocated_memory:.2f} GB")

Используемое устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Занято видеопамяти на GPU: 16.06 GB


In [ ]:
def create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples=None):

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"{prompt_config['entity']} information: "
            f"{row_to_text_template(row, feature_names, target_name, prompt_config)}\n"
            f"{prompt_config['question']}"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name, prompt_config)
            ex_target = prompt_config['pos_label'] if ex[target_name] == 1 else prompt_config['neg_label']

            messages.append({
                "role": "user",
                "content": f"{prompt_config['entity']} information: {ex_text}\n{prompt_config['question']}"
            })
            messages.append({
                "role": "assistant",
                "content": ex_target
            })

        client_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        messages.append({
            "role": "user",
            "content": f"{prompt_config['entity']} information: {client_text}\n{prompt_config['question']}"
        })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для меток из prompt_config
    first_token_logits = outputs.scores[0][0]

    pos_id = tokenizer.encode(prompt_config['pos_label'], add_special_tokens=False)[0]
    neg_id = tokenizer.encode(prompt_config['neg_label'], add_special_tokens=False)[0]

    pos_logit = first_token_logits[pos_id]
    neg_logit = first_token_logits[neg_id]

    probs = F.softmax(torch.stack([pos_logit, neg_logit]), dim=0)
    prob_pos = probs[0].item()

    return response, prob_pos

# Тест
test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, prob = predict_single_with_prob(test_prompt, prompt_config, base_model, tokenizer, device)
print(f"Response: '{response}', Probability: {prob:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Response: 'bad', Probability: 0.0052


# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
# @title
df_0 = train_df[train_df[target_name] == 0]
df_1 = train_df[train_df[target_name] == 1]

if len(df_0) > len(df_1):
    df_majority = df_0
    df_minority = df_1
else:
    df_majority = df_1
    df_minority = df_0

seed = 42
print(f"До балансировки:")
print(f"{prompt_config['neg_label']}:  {len(df_majority)}")
print(f"{prompt_config['pos_label']}: {len(df_minority)}")

# Oversample to match
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=seed
)

train_df_balanced = pd.concat([df_majority, df_minority_upsampled])
train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

print(f"\nПосле балансировки:")
print(train_df_balanced[target_name].value_counts())

До балансировки:
bad:  420
good: 180

После балансировки:
class
0    420
1    420
Name: count, dtype: int64


In [ ]:
# Создание текстов для обучения

def create_dataset(df):
    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )
    texts = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Подготовка данных"):
        input_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        target = prompt_config['pos_label']  if row[target_name] == 1 else prompt_config['neg_label']

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
            {"role": "assistant", "content": target}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return texts

train_texts = create_dataset(train_df_balanced)
train_dataset = Dataset.from_dict({"text": train_texts})

Подготовка данных:   0%|          | 0/840 [00:00<?, ?it/s]

In [ ]:
# Токенизация
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print(f"\n Подготовлено {len(tokenized_dataset)} примеров для обучения")

Map:   0%|          | 0/840 [00:00<?, ? examples/s]


 Подготовлено 840 примеров для обучения


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

num_epochs = 20
batch_size = 16

# Загрузка базовой модели
model_lora = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

# Настройка LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

# Аргументы для обучения
training_args_lora = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_torch_fused",
    warmup_steps=50,
    max_grad_norm=1.0,
    weight_decay=0.01,
    report_to="none",
    dataloader_num_workers=4,
    group_by_length=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Mounted at /content/drive


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Обучение
print(f"\nНачинаем обучение на {num_epochs} эпох")

train_start_time = time.time()
trainer_lora.train()
training_time = time.time() - train_start_time

print(f"Обучение завершено за {training_time/60:.1f} минут")

trainer_lora.save_model(output_dir)
print(f"Модель сохранена в {output_dir}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Начинаем обучение на 20 эпох


Step,Training Loss
10,2.179782
20,1.059478
30,0.249312
40,0.165635
50,0.136417
60,0.130102
70,0.126116
80,0.124595
90,0.124197
100,0.121711


Обучение завершено за 13.5 минут
Модель сохранена в /content/drive/MyDrive/finetuned_llama_lora_with_Credit_G


In [ ]:
import glob
import re
from peft import PeftModel
print("\nОценка checkpoint на val")

# Поиск и сортировка checkpoint
checkpoints = glob.glob(f"{output_dir}/checkpoint-*")
checkpoints_sorted = sorted(
    checkpoints,
    key=lambda x: int(re.search(r'checkpoint-(\d+)', x).group(1))
)

print(f"\nНайдено {len(checkpoints_sorted)} checkpoints\n")

# Хранение результатов
results_list = []
best_auc = 0.0
best_checkpoint = None

# Оценка каждого checkpoint
for checkpoint_path in checkpoints_sorted:
    checkpoint_name = checkpoint_path.split('/')[-1]
    checkpoint_num = int(re.search(r'checkpoint-(\d+)', checkpoint_name).group(1))

    # Загрузка модели
    model_eval = PeftModel.from_pretrained(base_model, checkpoint_path)
    model_eval.eval()

    # Оценка
    y_true, y_prob = [], []
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc=checkpoint_name):
        prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
        response, prob = predict_single_with_prob(prompt, prompt_config, model_eval, tokenizer,   device)
        y_true.append(row[target_name])
        y_prob.append(prob)

    # Метрики
    y_pred = [1 if p > 0.5 else 0 for p in y_prob]
    roc, f1, acc, pr, rec = compute_metrics(y_true, y_pred, y_prob)

    results_list.append({
        'Checkpoint': checkpoint_num,
        'ROC-AUC': roc,
        'F1': f1,
        'Accuracy': acc,
        'Precision': pr,
        'Recall': rec
    })

    if roc > best_auc:
        best_auc = roc
        best_checkpoint = checkpoint_path

    del model_eval
    torch.cuda.empty_cache()


Оценка checkpoint на val

Найдено 20 checkpoints



checkpoint-27:   0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


checkpoint-54:   0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


...

checkpoint-486:   0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


checkpoint-513:   0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


checkpoint-540:   0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
print(f"\nЛучший checkpoint: {best_checkpoint}")


Лучший checkpoint: /content/drive/MyDrive/finetuned_llama_lora_with_Credit_G/checkpoint-216


In [ ]:
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('Checkpoint')
results_df

,Checkpoint,ROC-AUC,F1,Accuracy,Precision,Recall
0,27,0.600774,0.000000,0.300,0.000000,0.000000
1,54,0.618452,0.000000,0.300,0.000000,0.000000
2,81,0.670417,0.750929,0.665,0.782946,0.721429
3,108,0.743333,0.820144,0.750,0.826087,0.814286
4,135,0.793452,0.834532,0.770,0.840580,0.828571
5,162,0.788571,0.260870,0.405,1.000000,0.150000
6,189,0.791726,0.800000,0.745,0.886957,0.728571
7,216,0.810417,0.742616,0.695,0.907216,0.628571
8,243,0.805833,0.704846,0.665,0.919540,0.571429
9,270,0.802738,0.761134,0.705,0.878505,0.671429


In [ ]:
model_finetuned = PeftModel.from_pretrained(base_model, best_checkpoint)
model_finetuned.eval()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
# Проверка качества обучения Fine-tuned модели

test_samples = {
    prompt_config['pos_label']: test_df[test_df[target_name] == 1].sample(n=10, random_state=42),
    prompt_config['neg_label']: test_df[test_df[target_name] == 0].sample(n=10, random_state=42)
}

# Проверка распределения вероятностей
print("\nРаспределение вероятностей")
all_probs = []
for _, row in test_df.sample(n=100, random_state=42).iterrows():
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
    _, prob = predict_single_with_prob(prompt, prompt_config, model_finetuned, tokenizer, device)
    all_probs.append(prob)

all_probs = np.array(all_probs)
print(f"P({prompt_config['pos_label']}) - Mean: {all_probs.mean():.4f}, Std: {all_probs.std():.4f}")
print(f"P({prompt_config['pos_label']}) - Min: {all_probs.min():.4f}, Max: {all_probs.max():.4f}")
print(f"P({prompt_config['pos_label']}) - Median: {np.median(all_probs):.4f}")


Распределение вероятностей
P(good) - Mean: 0.4655, Std: 0.1384
P(good) - Min: 0.2689, Max: 0.7311
P(good) - Median: 0.4073


In [ ]:
# Оценка на тестовой выборке
y_true_fine, y_pred_fine, y_prob_fine = [], [], []

start_time = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test evaluation"):
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
    response, prob = predict_single_with_prob(prompt, prompt_config, model_finetuned, tokenizer, device)
    prediction = parse_prediction(response, prompt_config)

    y_true_fine.append(row[target_name])
    y_pred_fine.append(prediction)
    y_prob_fine.append(prob)

fine_tuned_time = time.time() - start_time

print(f"fine_tuned_time: {fine_tuned_time}")
print(f"fine_tuned_time / len(y_true_fine) {fine_tuned_time / len(y_true_fine)}")


Test evaluation:   0%|          | 0/200 [00:00<?, ?it/s]

fine_tuned_time: 13.651411056518555
fine_tuned_time / len(y_true_fine) 0.06825705528259278


In [ ]:
# Вычисляем метрики

roc_fine, f1_fine, acc_fine, pr_fine, rec_fine = compute_metrics(
    np.array(y_true_fine),
    np.array(y_pred_fine),
    np.array(y_prob_fine)
)
print("Результаты fine-tune:")
print(f"ROC AUC: {roc_fine}")
print(f"F1 Score: {f1_fine}")
print(f"Accuracy: {acc_fine}")
print(f"Precision: {pr_fine}")
print(f"Recall: {rec_fine}")


Результаты fine-tune:
ROC AUC: 0.7285119047619047
F1 Score: 0.6484018264840182
Accuracy: 0.615
Precision: 0.8987341772151899
Recall: 0.5071428571428571


In [ ]:
fine_tuned_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_fine),
    np.array(y_pred_fine),
    np.array(y_prob_fine),
    n_iter=1000
)

print("\nРезультаты fine-tune (bootstrap метрики с доверительными интервалами):")
for key, value in fine_tuned_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты fine-tune (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.7277±0.0373
  F1: 0.6458±0.0382
  Accuracy: 0.6141±0.0349
  Precision: 0.8977±0.0355
  Recall: 0.5056±0.0427


In [ ]:
from google.colab import runtime
runtime.unassign()

Время обучения: 13 мин 30 сек

Время выборки моделя: ~ 5 мин

Время тестирования: ~ 13 сек

Использовано оперативная памяти графического процесса: 76.3/95.6GB.

Графический процессор G4 95.6GB.